# 11 · Leave-One-Dataset-Out — MRKR Held Out (3 seeds)

Trains the full method on OAI, NHANES III, and Mendeley, and evaluates on the held-out MRKR set across three random seeds.

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import sys, importlib
sys.path.insert(0,'/content/drive/MyDrive/Master Thesis/scope3')
import config; importlib.reload(config)
import pandas as pd, numpy as np
from pathlib import Path
import torch
if 'training_lib' in sys.modules: importlib.reload(sys.modules['training_lib'])
import training_lib as T
if not torch.cuda.is_available(): raise RuntimeError('No GPU. Runtime -> Change runtime type -> GPU.')
print('GPU:', torch.cuda.get_device_name(0))
manifest = T.prepare_local_data()
print(f'Manifest: {len(manifest):,} rows')

Mounted at /content/drive
GPU: NVIDIA A100-SXM4-40GB
Copied array in 33s
Loaded array (61558, 224, 224) in 1s
Manifest: 61,558 rows


In [2]:
def make_splits(manifest, test_dataset, seed=0):
    pool=manifest[manifest['dataset']!=test_dataset].copy()
    test=manifest[manifest['dataset']==test_dataset].copy()
    if 'split' in pool.columns and pool['split'].isin(['train','val']).any():
        tr=pool[pool['split'].isin(['train','TRAIN'])]; va=pool[pool['split'].isin(['val','VAL','validation'])]
        if len(va)==0: va=pool.sample(frac=0.15,random_state=seed); tr=pool.drop(va.index)
    else:
        va=pool.sample(frac=0.15,random_state=seed); tr=pool.drop(va.index)
    return tr.reset_index(drop=True), va.reset_index(drop=True), test.reset_index(drop=True)
print('Split helper ready.')

Split helper ready.


## Execution

In [3]:
fk='fold2_test_mrkr'; test_ds=config.LODO_FOLDS[fk]; mech=dict(config.FULL_METHOD)
try:
    _sw=pd.read_csv(str(config.RESULTS_DIR/'grl_lambda_sweep.csv'))
    _best=float(_sw.loc[_sw['external_acc5'].idxmax(),'grl_lambda_max'])
    mech['grl_lambda_max']=_best; print(f'Using swept GRL lambda: {_best}')
except Exception: print('No sweep file; using config GRL_LAMBDA_MAX')
results=[]
for seed in config.SEEDS:
    run=f'{fk}_seed{seed}'
    tr,va,te=make_splits(manifest,test_ds,seed=seed)
    print(f'--- {run} (train={len(tr):,} val={len(va):,} test={len(te):,}) ---',flush=True)
    r=T.run_training(run,tr,va,te,mech,seed,config.CKPT_DIR,config.RESULTS_DIR,log_fn=print)
    results.append(r)
df=pd.DataFrame(results)
print(f'External acc5: {df["external_acc5"].mean():.4f} +/- {df["external_acc5"].std():.4f}')
print(f'External QWK : {df["external_qwk"].mean():.4f}')
print(f'Gap          : {df["gap"].mean():.4f}')
df.to_csv(str(config.RESULTS_DIR/f'{fk}_results.csv'),index=False)

Using swept GRL lambda: 0.5
--- fold2_test_mrkr_seed0 (train=18,352 val=3,239 test=39,967) ---
Downloading: "https://download.pytorch.org/models/convnext_large-ea097f82.pth" to /root/.cache/torch/hub/checkpoints/convnext_large-ea097f82.pth


100%|██████████| 755M/755M [00:03<00:00, 231MB/s]


  [fold2_test_mrkr_seed0] ep1/18 loss=3.614 tr=0.419 val=0.436 gap=-0.017 qwk=0.115 grl=0.00 (64s)
  [fold2_test_mrkr_seed0] ep2/18 loss=2.963 tr=0.463 val=0.478 gap=-0.015 qwk=0.333 grl=0.14 (43s)
  [fold2_test_mrkr_seed0] ep3/18 loss=2.748 tr=0.486 val=0.477 gap=+0.009 qwk=0.350 grl=0.25 (43s)
  [fold2_test_mrkr_seed0] ep4/18 loss=2.645 tr=0.501 val=0.499 gap=+0.002 qwk=0.450 grl=0.34 (43s)
  [fold2_test_mrkr_seed0] ep5/18 loss=2.575 tr=0.505 val=0.513 gap=-0.008 qwk=0.508 grl=0.40 (43s)
  [fold2_test_mrkr_seed0] ep6/18 loss=2.533 tr=0.517 val=0.504 gap=+0.013 qwk=0.455 grl=0.44 (43s)
  [fold2_test_mrkr_seed0] ep7/18 loss=2.482 tr=0.519 val=0.520 gap=-0.001 qwk=0.526 grl=0.47 (43s)
  [fold2_test_mrkr_seed0] ep8/18 loss=2.438 tr=0.529 val=0.534 gap=-0.005 qwk=0.578 grl=0.48 (43s)
  [fold2_test_mrkr_seed0] ep9/18 loss=2.416 tr=0.533 val=0.537 gap=-0.004 qwk=0.557 grl=0.49 (43s)
  [fold2_test_mrkr_seed0] ep10/18 loss=2.401 tr=0.530 val=0.537 gap=-0.007 qwk=0.570 grl=0.49 (43s)
  [fold2_